In [1]:
import pandas as pd
import ast

# Load activation state sequences
sequences_path = "../results/preprocessing/activation_state_sequences.csv"

df = pd.read_csv(sequences_path)

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())

Dataset shape: (150, 5)

Columns:
['Prompt_ID', 'Response_Label', 'Layer_ID', 'State_Sequence', 'Sequence_Length']

First 5 rows:


,Prompt_ID,Response_Label,Layer_ID,State_Sequence,Sequence_Length
0,19,0,10,"[3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, ...",64
1,19,0,18,"[3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, ...",64
2,19,0,26,"[8, 8, 8, 0, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, ...",64
3,25,0,10,"[3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, ...",41
4,25,0,18,"[3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, ...",41


In [2]:
from sklearn.model_selection import train_test_split

# One row per prompt for creating the split
prompt_labels = (
    df[["Prompt_ID", "Response_Label"]]
    .drop_duplicates()
)

# Same split used in the original final evaluation
train_ids, test_ids = train_test_split(
    prompt_labels["Prompt_ID"],
    test_size=0.20,
    random_state=42,
    stratify=prompt_labels["Response_Label"]
)

train_ids = set(train_ids)
test_ids = set(test_ids)

# Create train and test sequence data
train_df = df[df["Prompt_ID"].isin(train_ids)].copy()
test_df = df[df["Prompt_ID"].isin(test_ids)].copy()

print("Train prompts:", len(train_ids))
print("Test prompts:", len(test_ids))

print("\nTrain labels:")
print(
    prompt_labels[
        prompt_labels["Prompt_ID"].isin(train_ids)
    ]["Response_Label"].value_counts().sort_index()
)

print("\nTest labels:")
print(
    prompt_labels[
        prompt_labels["Prompt_ID"].isin(test_ids)
    ]["Response_Label"].value_counts().sort_index()
)

print("\nTrain sequence rows:", len(train_df))
print("Test sequence rows:", len(test_df))

print("\nPrompt overlap:", len(train_ids.intersection(test_ids)))

print("\nTrain Prompt IDs:")
print(sorted(train_ids))

print("\nTest Prompt IDs:")
print(sorted(test_ids))

Train prompts: 40
Test prompts: 10

Train labels:
Response_Label
0    20
1    20
Name: count, dtype: int64

Test labels:
Response_Label
0    5
1    5
Name: count, dtype: int64

Train sequence rows: 120
Test sequence rows: 30

Prompt overlap: 0

Train Prompt IDs:
[19, 25, 52, 62, 75, 83, 84, 105, 111, 140, 177, 199, 224, 249, 265, 276, 282, 345, 352, 369, 376, 383, 394, 413, 429, 432, 487, 507, 513, 516, 523, 524, 579, 588, 621, 634, 646, 656, 724, 785]

Test Prompt IDs:
[164, 183, 310, 324, 347, 421, 488, 530, 637, 789]


In [3]:
from collections import Counter

# Use Layer 26 for the baseline comparison
layer = 26

train_layer26 = train_df[train_df["Layer_ID"] == layer].copy()
test_layer26 = test_df[test_df["Layer_ID"] == layer].copy()

# Convert State_Sequence from string to Python list
train_layer26["State_Sequence"] = train_layer26["State_Sequence"].apply(ast.literal_eval)
test_layer26["State_Sequence"] = test_layer26["State_Sequence"].apply(ast.literal_eval)

# Create state-frequency features
def create_state_frequency_features(data):
    rows = []

    for _, row in data.iterrows():
        sequence = row["State_Sequence"]
        total = len(sequence)

        counts = Counter(sequence)

        # Convert counts to proportions
        features = {
            f"State_{state}": counts.get(state, 0) / total
            for state in range(12)
        }

        features["Prompt_ID"] = row["Prompt_ID"]
        features["Response_Label"] = row["Response_Label"]

        rows.append(features)

    return pd.DataFrame(rows)


train_features = create_state_frequency_features(train_layer26)
test_features = create_state_frequency_features(test_layer26)

print("Train feature shape:", train_features.shape)
print("Test feature shape:", test_features.shape)

print("\nTrain features:")
display(train_features.head())

print("\nTest features:")
display(test_features.head())

Train feature shape: (40, 14)
Test feature shape: (10, 14)

Train features:


,State_0,State_1,State_2,State_3,State_4,State_5,State_6,State_7,State_8,State_9,State_10,State_11,Prompt_ID,Response_Label
0,0.171875,0.0,0.0,0.015625,0.031250,0.0,0.0,0.0,0.6875,0.046875,0.015625,0.03125,19,0
1,0.219512,0.0,0.0,0.121951,0.585366,0.0,0.0,0.0,0.0000,0.048780,0.024390,0.00000,25,0
2,0.506667,0.0,0.0,0.013333,0.440000,0.0,0.0,0.0,0.0000,0.026667,0.013333,0.00000,52,0
3,0.148148,0.0,0.0,0.296296,0.481481,0.0,0.0,0.0,0.0000,0.037037,0.037037,0.00000,62,1
4,0.487500,0.0,0.0,0.025000,0.462500,0.0,0.0,0.0,0.0000,0.025000,0.000000,0.00000,75,1



Test features:


,State_0,State_1,State_2,State_3,State_4,State_5,State_6,State_7,State_8,State_9,State_10,State_11,Prompt_ID,Response_Label
0,0.545455,0.0,0.090909,0.000000,0.272727,0.00,0.0,0.0,0.0,0.054545,0.018182,0.018182,164,0
1,0.558824,0.0,0.000000,0.000000,0.397059,0.00,0.0,0.0,0.0,0.029412,0.014706,0.000000,183,0
2,0.569231,0.0,0.369231,0.000000,0.000000,0.00,0.0,0.0,0.0,0.046154,0.015385,0.000000,310,1
3,0.074074,0.0,0.851852,0.018519,0.000000,0.00,0.0,0.0,0.0,0.037037,0.018519,0.000000,324,0
4,0.075000,0.0,0.000000,0.100000,0.000000,0.75,0.0,0.0,0.0,0.050000,0.025000,0.000000,347,1


In [4]:
# Separate input features (X) and target labels (y)

feature_columns = [f"State_{i}" for i in range(12)]

X_train = train_features[feature_columns]
y_train = train_features["Response_Label"]

X_test = test_features[feature_columns]
y_test = test_features["Response_Label"]

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

print("\nTraining labels:")
print(y_train.value_counts().sort_index())

print("\nTesting labels:")
print(y_test.value_counts().sort_index())

X_train shape: (40, 12)
X_test shape: (10, 12)
y_train shape: (40,)
y_test shape: (10,)

Training labels:
Response_Label
0    20
1    20
Name: count, dtype: int64

Testing labels:
Response_Label
0    5
1    5
Name: count, dtype: int64


In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# Create Logistic Regression model
lr_model = LogisticRegression(
    random_state=42,
    max_iter=1000
)

# Train the model
lr_model.fit(X_train, y_train)

# Predict on unseen test data
y_pred_lr = lr_model.predict(X_test)

# Calculate evaluation metrics
lr_accuracy = accuracy_score(y_test, y_pred_lr)
lr_precision = precision_score(y_test, y_pred_lr, zero_division=0)
lr_recall = recall_score(y_test, y_pred_lr, zero_division=0)
lr_f1 = f1_score(y_test, y_pred_lr, zero_division=0)

print("Logistic Regression Results")
print("---------------------------")
print(f"Accuracy : {lr_accuracy * 100:.1f}%")
print(f"Precision: {lr_precision * 100:.1f}%")
print(f"Recall   : {lr_recall * 100:.1f}%")
print(f"F1 Score : {lr_f1 * 100:.1f}%")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_lr))

Logistic Regression Results
---------------------------
Accuracy : 70.0%
Precision: 66.7%
Recall   : 80.0%
F1 Score : 72.7%

Confusion Matrix:
[[3 2]
 [1 4]]


In [6]:
from sklearn.ensemble import RandomForestClassifier

# Create Random Forest model
rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

# Train the model
rf_model.fit(X_train, y_train)

# Predict on unseen test data
y_pred_rf = rf_model.predict(X_test)

# Calculate evaluation metrics
rf_accuracy = accuracy_score(y_test, y_pred_rf)
rf_precision = precision_score(y_test, y_pred_rf, zero_division=0)
rf_recall = recall_score(y_test, y_pred_rf, zero_division=0)
rf_f1 = f1_score(y_test, y_pred_rf, zero_division=0)

print("Random Forest Results")
print("---------------------")
print(f"Accuracy : {rf_accuracy * 100:.1f}%")
print(f"Precision: {rf_precision * 100:.1f}%")
print(f"Recall   : {rf_recall * 100:.1f}%")
print(f"F1 Score : {rf_f1 * 100:.1f}%")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_rf))

Random Forest Results
---------------------
Accuracy : 70.0%
Precision: 100.0%
Recall   : 40.0%
F1 Score : 57.1%

Confusion Matrix:
[[5 0]
 [3 2]]


In [7]:
from sklearn.svm import SVC

# Create SVM model
svm_model = SVC(
    kernel="rbf",
    random_state=42
)

# Train the model
svm_model.fit(X_train, y_train)

# Predict on unseen test data
y_pred_svm = svm_model.predict(X_test)

# Calculate evaluation metrics
svm_accuracy = accuracy_score(y_test, y_pred_svm)
svm_precision = precision_score(y_test, y_pred_svm, zero_division=0)
svm_recall = recall_score(y_test, y_pred_svm, zero_division=0)
svm_f1 = f1_score(y_test, y_pred_svm, zero_division=0)

print("SVM Results")
print("-----------")
print(f"Accuracy : {svm_accuracy * 100:.1f}%")
print(f"Precision: {svm_precision * 100:.1f}%")
print(f"Recall   : {svm_recall * 100:.1f}%")
print(f"F1 Score : {svm_f1 * 100:.1f}%")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_svm))

SVM Results
-----------
Accuracy : 60.0%
Precision: 60.0%
Recall   : 60.0%
F1 Score : 60.0%

Confusion Matrix:
[[3 2]
 [2 3]]


In [8]:
# STEP 8: Final comparison table

comparison_results = pd.DataFrame({
    "Method": [
        "Sequential Pattern Mining (Proposed)",
        "Logistic Regression",
        "Random Forest",
        "SVM"
    ],
    "Accuracy": [
        70.0,
        lr_accuracy * 100,
        rf_accuracy * 100,
        svm_accuracy * 100
    ],
    "Precision": [
        66.7,
        lr_precision * 100,
        rf_precision * 100,
        svm_precision * 100
    ],
    "Recall": [
        80.0,
        lr_recall * 100,
        rf_recall * 100,
        svm_recall * 100
    ],
    "F1": [
        72.7,
        lr_f1 * 100,
        rf_f1 * 100,
        svm_f1 * 100
    ]
})

# Round values for clean presentation
comparison_results[
    ["Accuracy", "Precision", "Recall", "F1"]
] = comparison_results[
    ["Accuracy", "Precision", "Recall", "F1"]
].round(1)

display(comparison_results)

,Method,Accuracy,Precision,Recall,F1
0,Sequential Pattern Mining (Proposed),70.0,66.7,80.0,72.7
1,Logistic Regression,70.0,66.7,80.0,72.7
2,Random Forest,70.0,100.0,40.0,57.1
3,SVM,60.0,60.0,60.0,60.0


In [11]:
# STEP 9: Verify proposed method predictions against baseline predictions

# Load the original saved held-out evaluation results
proposed_results = pd.read_csv(
    "../results/final_evaluation_outputs/held_out_test_results.csv"
)

# Rename the proposed warning prediction column
proposed_results = proposed_results.rename(
    columns={"Warning": "Proposed_Pattern_Mining"}
)

print("Saved proposed-method test results:")
display(proposed_results)

Saved proposed-method test results:


,Prompt_ID,Actual_Label,Proposed_Pattern_Mining,Warning_Fraction
0,164,0,1,0.981818
1,183,0,0,NaN
2,310,1,0,NaN
3,324,0,0,NaN
4,347,1,1,0.625000
5,421,1,1,0.750000
6,488,0,1,0.941176
7,530,0,0,NaN
8,637,1,1,0.150000
9,789,1,1,0.212121


In [12]:
# Verify the proposed method's saved metrics

proposed_y_true = proposed_results["Actual_Label"]
proposed_y_pred = proposed_results["Proposed_Pattern_Mining"]

print("Proposed Sequential Pattern Mining")
print("----------------------------------")
print(f"Accuracy : {accuracy_score(proposed_y_true, proposed_y_pred) * 100:.1f}%")
print(f"Precision: {precision_score(proposed_y_true, proposed_y_pred, zero_division=0) * 100:.1f}%")
print(f"Recall   : {recall_score(proposed_y_true, proposed_y_pred, zero_division=0) * 100:.1f}%")
print(f"F1 Score : {f1_score(proposed_y_true, proposed_y_pred, zero_division=0) * 100:.1f}%")

print("\nConfusion Matrix:")
print(confusion_matrix(proposed_y_true, proposed_y_pred))

Proposed Sequential Pattern Mining
----------------------------------
Accuracy : 70.0%
Precision: 66.7%
Recall   : 80.0%
F1 Score : 72.7%

Confusion Matrix:
[[3 2]
 [1 4]]


In [13]:
# STEP 10: Save final baseline comparison results

comparison_results.to_csv(
    "../results/final_evaluation_outputs/baseline_comparison_results.csv",
    index=False
)

print("Baseline comparison results saved successfully.")
print("\nFinal Comparison:")
display(comparison_results)

Baseline comparison results saved successfully.

Final Comparison:


,Method,Accuracy,Precision,Recall,F1
0,Sequential Pattern Mining (Proposed),70.0,66.7,80.0,72.7
1,Logistic Regression,70.0,66.7,80.0,72.7
2,Random Forest,70.0,100.0,40.0,57.1
3,SVM,60.0,60.0,60.0,60.0


In [14]:
# STEP 11: Display confusion matrices for all baseline models

print("Logistic Regression Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_lr))

print("\nRandom Forest Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_rf))

print("\nSVM Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_svm))

Logistic Regression Confusion Matrix:
[[3 2]
 [1 4]]

Random Forest Confusion Matrix:
[[5 0]
 [3 2]]

SVM Confusion Matrix:
[[3 2]
 [2 3]]


In [15]:
# STEP 12: Classification reports for baseline models

from sklearn.metrics import classification_report

print("===== Logistic Regression =====")
print(classification_report(
    y_test,
    y_pred_lr,
    target_names=["Truthful", "Hallucinated"],
    zero_division=0
))

print("===== Random Forest =====")
print(classification_report(
    y_test,
    y_pred_rf,
    target_names=["Truthful", "Hallucinated"],
    zero_division=0
))

print("===== SVM =====")
print(classification_report(
    y_test,
    y_pred_svm,
    target_names=["Truthful", "Hallucinated"],
    zero_division=0
))

===== Logistic Regression =====
              precision    recall  f1-score   support

    Truthful       0.75      0.60      0.67         5
Hallucinated       0.67      0.80      0.73         5

    accuracy                           0.70        10
   macro avg       0.71      0.70      0.70        10
weighted avg       0.71      0.70      0.70        10

===== Random Forest =====
              precision    recall  f1-score   support

    Truthful       0.62      1.00      0.77         5
Hallucinated       1.00      0.40      0.57         5

    accuracy                           0.70        10
   macro avg       0.81      0.70      0.67        10
weighted avg       0.81      0.70      0.67        10

===== SVM =====
              precision    recall  f1-score   support

    Truthful       0.60      0.60      0.60         5
Hallucinated       0.60      0.60      0.60         5

    accuracy                           0.60        10
   macro avg       0.60      0.60      0.60        10


In [16]:
# STEP 13: Save the final comparison table with all methods

final_comparison = comparison_results.copy()

final_comparison.to_csv(
    "../results/final_evaluation_outputs/final_method_comparison.csv",
    index=False
)

print("Final method comparison saved successfully.")
display(final_comparison)

Final method comparison saved successfully.


,Method,Accuracy,Precision,Recall,F1
0,Sequential Pattern Mining (Proposed),70.0,66.7,80.0,72.7
1,Logistic Regression,70.0,66.7,80.0,72.7
2,Random Forest,70.0,100.0,40.0,57.1
3,SVM,60.0,60.0,60.0,60.0
